# Phase 2 — Similarity search baseline

Mount Drive, read the audited 20K corpus, create 100 deterministic query texts, build TF-IDF vectors, and benchmark a **naive Python cosine top-k** against a NumPy/scikit-learn reference. Timings exclude vectorization, file I/O, and query creation. Start with 1K before scaling to 5K, 10K and 20K. Save raw timings to Drive.

In [ ]:
%pip -q install scikit-learn pandas

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import json, time, csv, random, statistics, cProfile, pstats, io
from pathlib import Path
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
ROOT=Path('/content/drive/MyDrive/HP-SS-Engine')
RAW=ROOT/'data/raw/qwen_corpus_20k_progress.jsonl'
OUT=ROOT/'benchmarks/raw_results'; OUT.mkdir(parents=True,exist_ok=True)
rows=[json.loads(s) for s in RAW.open(encoding='utf-8') if s.strip()]
assert len(rows)==20000 and {r['id'] for r in rows}==set(range(20000))
rows.sort(key=lambda x:x['id'])
texts=[r['response'] for r in rows]
query_ids=list(range(0,20000,200))[:100]
queries=[rows[i]['prompt'] for i in query_ids]
assert len(queries)==100
print('Documents:',len(texts),'queries:',len(queries))

In [ ]:
# TF-IDF is a sparse CSR representation. Cosine similarity equals dot product for L2-normalized vectors.
vectorizer=TfidfVectorizer(dtype=np.float32,norm='l2',min_df=1)
X=vectorizer.fit_transform(texts)
Q=vectorizer.transform(queries)
print('TF-IDF shape:',X.shape,'nonzeros:',X.nnz,'density:',X.nnz/(X.shape[0]*X.shape[1]))


In [ ]:
def naive_python_cosine_topk(docs_csr, query_csr, k=10):
    # Intentionally naive: Python document loop, sparse dot per document.
    scores=[]
    for i in range(docs_csr.shape[0]):
        score=float(docs_csr.getrow(i).multiply(query_csr).sum())
        scores.append((score,i))
    return sorted(scores,key=lambda x:(-x[0],x[1]))[:k]

def reference_topk(docs_csr, query_csr, k=10):
    scores=(docs_csr@query_csr.T).toarray().ravel()
    order=np.lexsort((np.arange(len(scores)),-scores))[:k]
    return [(float(scores[i]),int(i)) for i in order]

for n in (100,1000):
    actual=naive_python_cosine_topk(X[:n],Q[0],10)
    expected=reference_topk(X[:n],Q[0],10)
    assert [i for _,i in actual]==[i for _,i in expected]
    assert np.allclose([s for s,_ in actual],[s for s,_ in expected],atol=1e-5)
print('Correctness check passed for 100 and 1,000 documents')

In [ ]:
# Profile one 1K search to identify dominant functions before optimization.
prof=cProfile.Profile();prof.enable();naive_python_cosine_topk(X[:1000],Q[0],10);prof.disable()
buf=io.StringIO();pstats.Stats(prof,stream=buf).sort_stats('cumtime').print_stats(20)
print(buf.getvalue())
(ROOT/'benchmarks/profiling').mkdir(parents=True,exist_ok=True)
(ROOT/'benchmarks/profiling/naive_1k_cprofile.txt').write_text(buf.getvalue(),encoding='utf-8')

In [ ]:
# Naive baseline is expensive. Run 1K first; uncomment larger sizes only after checking runtime.
SCALES=[1000] # Later: [1000,5000,10000,20000]
TIMED_QUERIES=5 # Increase after confirming runtime; query set has 100 items.
REPEATS=5
results=[]
for n in SCALES:
    docs=X[:n]
    naive_python_cosine_topk(docs,Q[0],10) # warmup
    for qi in range(TIMED_QUERIES):
        ref=reference_topk(docs,Q[qi],10)
        for rep in range(REPEATS):
            t0=time.perf_counter_ns();got=naive_python_cosine_topk(docs,Q[qi],10);dt=time.perf_counter_ns()-t0
            assert [i for _,i in got]==[i for _,i in ref]
            results.append({'implementation':'python_sparse_naive','n_docs':n,'query_index':qi,'repeat':rep,'top_k':10,'elapsed_ms':dt/1e6,'representation':'tfidf_csr_float32'})
    print(n,'median latency ms:',round(statistics.median(r['elapsed_ms'] for r in results if r['n_docs']==n),3))
path=OUT/'naive_cosine_tfidf_raw.csv'
with path.open('w',newline='',encoding='utf-8') as f:
    writer=csv.DictWriter(f,fieldnames=list(results[0]));writer.writeheader();writer.writerows(results)
print('Saved:',path,'measurements:',len(results))